In [1]:
import gc

In [2]:
import numpy as np
import xarray as xr

In [3]:
from data import (
    ds_single,
    ds_pressure,
    _load_single_dataset,
    _load_pressure_dataset,
)

In [4]:
import fcbench

In [5]:
from tqdm import tqdm

In [6]:
def compute_absolute_error_bound(
    v: xr.DataArray, time_chunk: int = 25
) -> tuple[float, float]:
    vrange = float(np.nanmax(v) - np.nanmin(v))

    if vrange == 0.0:
        return (0.0, 1.0)

    prec = int(np.ceil(np.log10(vrange)))
    all_us = [0.0]

    while True:
        us = []

        for i in range(0, v.valid_time.size, time_chunk):
            vi = v.isel(valid_time=slice(i, i + time_chunk)).compute()

            v_linquant = np.round(vi / (10**prec)) * (10**prec)
            v_err = v_linquant - vi

            us.append(fcbench.metrics.Uniformity.goodness(v_err))

            break

        all_us.append(np.amin(us))

        if len(all_us) >= 5 and all(p <= all_us[-5] for p in all_us[-5:]):
            return 10 ** (prec + 4), all_us[-5]

        prec -= 1

In [7]:
with open("uniformity-single.txt", "w") as f:
    for i, key in tqdm(enumerate(sorted(ds_single.mean))):
        if (i > 0) and (i % 50 == 0):
            del ds_single
            gc.collect()
            ds_single = _load_single_dataset()
        print(f"{key}: ", end="", flush=True, file=f)
        print(
            f"{compute_absolute_error_bound(ds_single.mean[key])!r}", flush=True, file=f
        )

226it [22:59,  6.11s/it]


In [8]:
with open("uniformity-pressure.txt", "w") as f:
    for i, key in tqdm(enumerate(sorted(ds_pressure.mean)), position=0):
        if (i > 0) and (i % 8 == 0):
            del ds_pressure
            gc.collect()
            ds_pressure = _load_pressure_dataset()
        print(f"{key}:", flush=True, file=f)
        for pressure in tqdm(
            sorted(ds_pressure.mean[key].pressure_level.values), position=1
        ):
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(
                f"{compute_absolute_error_bound(ds_pressure.mean[key].sel(pressure_level=pressure))!r}",
                flush=True,
                file=f,
            )

0it [00:00, ?it/s]
%|                                                                    | 0/4 [00:00<?, ?it/s]
%|███████████████                                             | 1/4 [00:04<00:14,  4.83s/it]
%|██████████████████████████████                              | 2/4 [00:09<00:09,  4.62s/it]
%|█████████████████████████████████████████████               | 3/4 [00:14<00:04,  4.79s/it]
100%|████████████████████████████████████████████████████████████| 4/4 [00:18<00:00,  4.58s/it]
1it [00:18, 18.30s/it]
%|                                                                    | 0/4 [00:00<?, ?it/s]
%|███████████████                                             | 1/4 [00:05<00:16,  5.49s/it]
%|██████████████████████████████                              | 2/4 [00:11<00:11,  5.70s/it]
%|█████████████████████████████████████████████               | 3/4 [00:16<00:05,  5.38s/it]
100%|████████████████████████████████████████████████████████████| 4/4 [00:23<00:00,  5.85s/it]
2it [00:41, 21.30s/it]